## DATASET PREPARATION

In [ ]:

import re
import os
import pandas as pd

files = ["season1.txt", "season2.txt"]

all_rows = []



def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()




def get_startup_name(text):

    patterns = [

        r'owner of ([A-Za-z0-9\'\-\s&]+)',
        r'founder of ([A-Za-z0-9\'\-\s&]+)',
        r'president and c\.E\.O\. and founder of ([A-Za-z0-9\'\-\s&]+)',
        r'my product is ([A-Za-z0-9\'\-\s&]+)',
        r'business is ([A-Za-z0-9\'\-\s&]+)',
        r'company is ([A-Za-z0-9\'\-\s&]+)',
        r'next greatest food brand-- ([A-Za-z0-9\'\-\s&]+)'
    ]

    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            name = m.group(1).strip()
            name = name[:60]
            return name.title()

    return "Unknown"



def get_funding(text):

    patterns = [

        r'\$([\d,]+)\s*(?:for|in exchange for).*?(\d+)\%',
        r'asking you for\s*\$([\d,]+).*?(\d+)\%',
        r'seeking\s*\$([\d,]+).*?(\d+)\%',
    ]

    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)

        if m:
            amount = "$" + m.group(1)
            equity = m.group(2) + "%"
            return amount, equity

    return "", ""




def get_questions(text):

    qs = re.findall(r'([A-Z][^?.!]{5,120}\?)', text)

    cleaned = []

    for q in qs:

        bad_words = [
            "who are the sharks",
            "i need money",
            "what i want"
        ]

        if not any(b in q.lower() for b in bad_words):
            cleaned.append(q.strip())

    return cleaned[:8]



def get_criticisms(text):

    keywords = [
        "too high",
        "too rich",
        "not a business",
        "greedy",
        "crazy",
        "problem",
        "risk",
        "you are out of your mind",
        "i'm out",
        "no way"
    ]

    sents = re.split(r'[.!?]', text)

    out = []

    for s in sents:
        for kw in keywords:
            if kw in s.lower():
                out.append(s.strip())
                break

    return out[:8]



def get_decision(text):

    tail = text[int(len(text)*0.75):].lower()

    if any(x in tail for x in [
        "i'll take it",
        "we have a deal",
        "deal accepted",
        "i accept",
        "i'd love to take your offer"
    ]):
        return "Deal"

    if tail.count("i'm out") >= 2 or "all out" in tail or "no deal" in tail:
        return "Rejected"

    return "Unknown"




def parse_episode(text, season, episode):

    text = clean_text(text)

    row = {
        "season": season,
        "episode": episode,
        "startup_name": get_startup_name(text),
        "ask_amount": get_funding(text)[0],
        "equity": get_funding(text)[1],
        "questions": " | ".join(get_questions(text)),
        "criticisms": " | ".join(get_criticisms(text)),
        "final_decision": get_decision(text)
    }

    return row



for file in files:

    if not os.path.exists(file):
        print(file, "not found")
        continue

    with open(file, "r", encoding="latin-1") as f:
        data = f.read()

    headers = re.findall(
        r'===== SEASON\s+(\d+)\s+EPISODE\s+(\d+)\s+=====',
        data
    )

    blocks = re.split(
        r'===== SEASON\s+\d+\s+EPISODE\s+\d+\s+=====',
        data
    )[1:]

    for i in range(len(blocks)):

        season = headers[i][0]
        episode = headers[i][1]

        row = parse_episode(blocks[i], season, episode)
        all_rows.append(row)



df = pd.DataFrame(all_rows)

df.to_csv("sharktank_clean_dataset.csv", index=False)

print("Saved Successfully!")
print(df.shape)
print(df.head(20))

Saved Successfully!
(15, 8)
   season episode                                       startup_name  \
0       1       1                                                 Mr   
1       1       2  Gonna Revolutionize The Way Things Are Taught ...   
2       1       3                                    The Turbobaster   
3       1       4                          Graffiti Removal Services   
4       1       5                                    Called The Fizz   
5       1       7            The Solution To Every Griller'S Problem   
6       1       8                                   Probably - Worth   
7       1       9             The Gayla Bentley Fashion Design Group   
8       1      10                         Called Pillars Of Slippers   
9       1      11                                     The Factionist   
10      1      12                           Worth What You'Re Asking   
11      2       1                               The Dallas Mavericks   
12      2       2                   

In [ ]:
import pandas as pd
import re

df = pd.read_csv("sharktank_clean_dataset.csv")



bad_names = [
    "Mr",
    "Probably - Worth",
    "Worth What You'Re Asking",
    "Actually Getting The Product To The Customer"
]

df["startup_name"] = df["startup_name"].replace(
    {
        "Mr": "Mr Tod's Pie Factory",
        "Probably - Worth": "Unknown",
        "Worth What You'Re Asking": "Unknown",
        "Actually Getting The Product To The Customer": "Unknown"
    }
)


def fix_equity(x):
    try:
        val = int(str(x).replace("%",""))
        if val > 100:
            return "Unknown"
        return str(val) + "%"
    except:
        return "Unknown"

df["equity"] = df["equity"].apply(fix_equity)


def clean_questions(q):

    bad = [
        "I want ?",
        "What I want ?",
        "Who are the sharks?"
    ]

    parts = str(q).split("|")

    good = []

    for p in parts:
        p = p.strip()
        if p not in bad and len(p) > 8:
            good.append(p)

    return " | ".join(good[:5])

df["questions"] = df["questions"].apply(clean_questions)


df.drop_duplicates(inplace=True)



df.to_csv("sharktank_final_cleaned.csv", index=False)

print(df[["startup_name","equity","final_decision"]])
print(df.shape)

                                         startup_name   equity final_decision
0                                Mr Tod's Pie Factory      10%       Rejected
1   Gonna Revolutionize The Way Things Are Taught ...      10%           Deal
2                                     The Turbobaster      30%       Rejected
3                           Graffiti Removal Services      40%        Unknown
4                                     Called The Fizz      50%       Rejected
5             The Solution To Every Griller'S Problem      25%       Rejected
6                                             Unknown      10%       Rejected
7              The Gayla Bentley Fashion Design Group      10%           Deal
8                          Called Pillars Of Slippers      20%       Rejected
9                                      The Factionist  Unknown           Deal
10                                            Unknown  Unknown       Rejected
11                               The Dallas Mavericks      10%  

In [ ]:
import pdfplumber
import os
import pandas as pd

!pip install pdfplumber

folder = "pitch_decks/"
rows = []


for file in os.listdir(folder):
    if file.endswith(".pdf"):
        path = os.path.join(folder, file)

        with pdfplumber.open(path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()

                if text:
                    rows.append({
                        "startup": file.replace(".pdf",""),
                        "slide_no": i+1,
                        "slide_text": text
                    })

df = pd.DataFrame(rows)
df.to_csv("pitchdeck_slides.csv", index=False)
print(df.shape)

(24, 3)


In [ ]:
print(df.head())

                  startup  slide_no  \
0  CrowdBouncerPitch-Deck         1   
1  CrowdBouncerPitch-Deck         2   
2  CrowdBouncerPitch-Deck         3   
3  CrowdBouncerPitch-Deck         4   
4  CrowdBouncerPitch-Deck         5   

                                          slide_text  
0  Pitch Deck\nFebruary 2014\nThis document conta...  
1  The History\nSeptember 2013: October 2013:\nMa...  
2  The Problem\nRaising money over the Internet i...  
3  The Current Platform\nCrowdBouncer is the Comp...  
4  Business Model & Financials\n• Charge for Veri...  


In [ ]:

import pandas as pd
import random

random.seed(42)



instructions_1 = [
    "Critique startup fundraising plan",
    "Evaluate investor readiness",
    "Review startup pitch",
    "Assess startup valuation",
    "Explain investor concern",
    "Provide VC feedback",
    "Analyze fundraising strategy",
    "Evaluate business model",
    "Check market attractiveness",
    "Assess startup moat"
]

inputs_1 = [
    "We are raising at $20M valuation pre-revenue",
    "We target all businesses globally",
    "We have no clear revenue model",
    "Large market opportunity",
    "No competitors exist",
    "We only need funding for marketing",
    "We have MVP but no paying customers",
    "We will grow through social media only",
    "We are another food delivery app",
    "Need $5M seed round with no traction",
    "We have 100 users after 2 years",
    "We burn $200k monthly with no growth",
    "We have no CAC or LTV numbers",
    "Our founders have no domain experience",
    "We rely on one enterprise customer"
]

outputs_1 = {
    "We are raising at $20M valuation pre-revenue":
        "Valuation appears aggressive without revenue, traction, or clear growth signals.",

    "We target all businesses globally":
        "ICP is too broad. Investors prefer focused customer segments with clear pain points.",

    "We have no clear revenue model":
        "Lack of monetization clarity increases investor hesitation and execution risk.",

    "Large market opportunity":
        "Too vague. Add TAM, SAM, SOM estimates with assumptions and sources.",

    "No competitors exist":
        "Usually unrealistic. Show direct, indirect, or substitute competitors honestly.",

    "We only need funding for marketing":
        "Use of funds should include growth, product, hiring, and milestones.",

    "We have MVP but no paying customers":
        "May suit angel funding, but many VCs will seek stronger validation first.",

    "We will grow through social media only":
        "Go-to-market strategy is shallow. Need repeatable acquisition channels.",

    "We are another food delivery app":
        "Crowded category. Need clear differentiation, defensibility, or niche advantage.",

    "Need $5M seed round with no traction":
        "Capital ask seems high relative to stage. Justify with milestones and roadmap.",

    "We have 100 users after 2 years":
        "Growth appears slow. Explain retention, niche strategy, or learning progress.",

    "We burn $200k monthly with no growth":
        "High burn without traction is a major investor red flag.",

    "We have no CAC or LTV numbers":
        "Missing unit economics weakens confidence in scalable growth.",

    "Our founders have no domain experience":
        "Need stronger founder-market fit, advisors, or evidence of execution capability.",

    "We rely on one enterprise customer":
        "Customer concentration risk is high. Revenue diversification needed."
}

rows1 = []

N1 = 1200   

for _ in range(N1):
    inp = random.choice(inputs_1)
    row = {
        "instruction": random.choice(instructions_1),
        "input": inp,
        "output": outputs_1[inp]
    }
    rows1.append(row)

df1 = pd.DataFrame(rows1)
df1.to_csv("synthetic_vc_logic.csv", index=False)

print("Saved synthetic_vc_logic.csv")
print(df1.shape)

Saved synthetic_vc_logic.csv
(1200, 3)


In [ ]:


instructions_2 = [
    "Rewrite startup slide",
    "Improve pitch deck slide",
    "Rewrite weak startup statement",
    "Make this slide investor ready",
    "Strengthen startup pitch slide"
]

inputs_2 = [
    "Large market opportunity",
    "We help businesses grow",
    "Need funding",
    "We have users",
    "Great team",
    "Many competitors exist",
    "Subscription model",
    "Our product is unique",
    "We target everyone",
    "We solve a problem",
    "Good traction",
    "We are growing fast",
    "Strong margins",
    "Need to hire people",
    "We use AI"
]

outputs_2 = {
    "Large market opportunity":
        "$12B global SaaS market growing at 18% CAGR with rising SME digitization demand.",

    "We help businesses grow":
        "AI-powered CRM platform helping SMEs increase conversion rates by 32%.",

    "Need funding":
        "Raising $500k to accelerate GTM, hire engineers, and expand pilot customers.",

    "We have users":
        "10,000 monthly active users with 22% month-over-month growth.",

    "Great team":
        "Founding team from Google, McKinsey, and IIT with deep domain expertise.",

    "Many competitors exist":
        "Fragmented market with no clear leader, creating room for category dominance.",

    "Subscription model":
        "$29/month SaaS pricing with annual enterprise plans and 78% gross margins.",

    "Our product is unique":
        "Patent-pending AI workflow engine reducing manual effort by 60%.",

    "We target everyone":
        "Initial focus on Indian SMEs with 20–200 employees in retail and services.",

    "We solve a problem":
        "Eliminates 8+ hours/week of manual reconciliation for finance teams.",

    "Good traction":
        "$40k MRR with 9% monthly growth and 85% customer retention.",

    "We are growing fast":
        "Revenue grew 4.2x in the last 12 months through organic demand.",

    "Strong margins":
        "Gross margins of 76% with improving payback period below 5 months.",

    "Need to hire people":
        "Hiring senior engineering and sales leaders to scale product and revenue.",

    "We use AI":
        "Uses proprietary AI models to automate decision workflows in real time."
}

rows2 = []

N2 = 1200

for _ in range(N2):
    inp = random.choice(inputs_2)
    row = {
        "instruction": random.choice(instructions_2),
        "input": inp,
        "output": outputs_2[inp]
    }
    rows2.append(row)

df2 = pd.DataFrame(rows2)
df2.to_csv("synthetic_slide_rewrites.csv", index=False)

print("Saved synthetic_slide_rewrites.csv")
print(df2.shape)


Saved synthetic_slide_rewrites.csv
(1200, 3)


In [ ]:

!git clone https://huggingface.co/datasets/takala/financial_phrasebank

Cloning into 'financial_phrasebank'...
remote: Enumerating objects: 75, done.
remote: Total 75 (delta 0), reused 0 (delta 0), pack-reused 75 (from 1)
Receiving objects: 100% (75/75), 20.71 KiB | 6.90 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [ ]:


import os
import pandas as pd

folder = "financial_phrasebank"

print("Files inside repo:")
print(os.listdir(folder))

Files inside repo:
['.git', 'financial_phrasebank.py', 'data', 'README.md', '.gitattributes']


In [ ]:
import pandas as pd

file_path = "/content/financial_phrasebank/data/Sentences_AllAgree.txt"

rows = []

with open(file_path, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()

        if "@" in line:
            sentence, label = line.rsplit("@", 1)

            rows.append({
                "sentence": sentence.strip(),
                "label": label.strip()
            })

df = pd.DataFrame(rows)

print(df.head())
print(df.shape)

df.to_csv("financial_phrasebank_allagree.csv", index=False)

print("Saved Successfully!")

                                            sentence     label
0  According to Gran , the company has no plans t...   neutral
1  For the last quarter of 2010 , Componenta 's n...  positive
2  In the third quarter of 2010 , net sales incre...  positive
3  Operating profit rose to EUR 13.1 mn from EUR ...  positive
4  Operating profit totalled EUR 21.1 mn , up fro...  positive
(2264, 2)
Saved Successfully!


In [ ]:

import pandas as pd
import re



def clean_text(x):
    x = str(x)
    x = re.sub(r'\s+', ' ', x)
    return x.strip()

all_rows = []

pitch = pd.read_csv("/content/pitchdeck_slides.csv")

for _, row in pitch.iterrows():

    txt = clean_text(row["slide_text"])

    if len(txt) < 20:
        continue

    all_rows.append({
        "instruction": "Analyze this startup pitch deck slide and provide investor feedback",
        "input": txt,
        "output": "Evaluate clarity, traction, market size, differentiation, and funding readiness."
    })



shark = pd.read_csv("/content/sharktank_final_cleaned.csv")

for _, row in shark.iterrows():

    inp = f"""
Startup: {row['startup_name']}
Ask Amount: {row['ask_amount']}
Equity: {row['equity']}
"""

    out = f"""
Questions: {row['questions']}
Criticisms: {row['criticisms']}
Final Decision: {row['final_decision']}
"""

    all_rows.append({
        "instruction": "Evaluate this Shark Tank startup pitch",
        "input": clean_text(inp),
        "output": clean_text(out)
    })


vc = pd.read_csv("/content/synthetic_vc_logic.csv")

for _, row in vc.iterrows():

    all_rows.append({
        "instruction": clean_text(row["instruction"]),
        "input": clean_text(row["input"]),
        "output": clean_text(row["output"])
    })


rw = pd.read_csv("/content/synthetic_slide_rewrites.csv")

for _, row in rw.iterrows():

    all_rows.append({
        "instruction": clean_text(row["instruction"]),
        "input": clean_text(row["input"]),
        "output": clean_text(row["output"])
    })



fin = pd.read_csv("/content/financial_phrasebank_allagree.csv")

for _, row in fin.iterrows():

    all_rows.append({
        "instruction": "Classify sentiment of this financial statement",
        "input": clean_text(row["sentence"]),
        "output": clean_text(row["label"])
    })




master = pd.DataFrame(all_rows)
 
master.drop_duplicates(inplace=True)

master = master[
    (master["input"].str.len() > 5) &
    (master["output"].str.len() > 3)
]

 
master = master.sample(frac=1, random_state=42).reset_index(drop=True)



master.to_csv("final_training_dataset.csv", index=False)

master.to_json(
    "final_training_dataset.jsonl",
    orient="records",
    lines=True
)

print("Saved Successfully!")
print(master.shape)
print(master.head(10))

Saved Successfully!
(2521, 3)
                                      instruction  \
0                  Rewrite weak startup statement   
1  Classify sentiment of this financial statement   
2  Classify sentiment of this financial statement   
3  Classify sentiment of this financial statement   
4  Classify sentiment of this financial statement   
5                        Assess startup valuation   
6  Classify sentiment of this financial statement   
7  Classify sentiment of this financial statement   
8  Classify sentiment of this financial statement   
9  Classify sentiment of this financial statement   

                                               input  \
0                                         Great team   
1  JP Morgan expects that Scala will lower Nobel ...   
2  Last week , the Finnish metals and technology ...   
3  First quarter underlying operating profit rose...   
4  The company has a continuous need for alloys s...   
5             We will grow through social media on

In [ ]:
import pandas as pd

df = pd.read_csv("final_training_dataset.csv")

print(df.shape)
print(df.head())
print(df.isnull().sum())
print(df["instruction"].value_counts())

(2521, 3)
                                      instruction  \
0                  Rewrite weak startup statement   
1  Classify sentiment of this financial statement   
2  Classify sentiment of this financial statement   
3  Classify sentiment of this financial statement   
4  Classify sentiment of this financial statement   

                                               input  \
0                                         Great team   
1  JP Morgan expects that Scala will lower Nobel ...   
2  Last week , the Finnish metals and technology ...   
3  First quarter underlying operating profit rose...   
4  The company has a continuous need for alloys s...   

                                              output  
0  Founding team from Google, McKinsey, and IIT w...  
1                                           negative  
2                                           positive  
3                                           positive  
4                                            neutral  
inst

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split


df = pd.read_csv("final_training_dataset.csv")

print("Original Shape:", df.shape)




required_cols = ["instruction", "input", "output"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")



def clean_text(text):
    text = str(text)

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    
    text = text.strip()

    return text


df["instruction"] = df["instruction"].apply(clean_text)
df["input"] = df["input"].apply(clean_text)
df["output"] = df["output"].apply(clean_text)


df.dropna(inplace=True)

df = df[
    (df["instruction"].str.len() > 5) &
    (df["input"].str.len() > 5) &
    (df["output"].str.len() > 3)
]



before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)

duplicates_removed = before - after

Original Shape: (2521, 3)


In [ ]:
# ADD TASK TYPE TAGGING (Application-Specific Quality Check)

def detect_task(instr):

    x = instr.lower()

    if "rewrite" in x:
        return "rewrite"

    elif "classify sentiment" in x:
        return "classification"

    elif "shark tank" in x:
        return "investor_eval"

    elif "pitch deck" in x:
        return "pitch_analysis"

    elif "critique" in x:
        return "critique"

    else:
        return "general"

df["task_type"] = df["instruction"].apply(detect_task)



print("\nCleaned Shape:", df.shape)
print("\nTask Distribution:")
print(df["task_type"].value_counts())

print("\nAverage Input Length:",
      round(df["input"].str.len().mean(), 2))

print("Average Output Length:",
      round(df["output"].str.len().mean(), 2))



Cleaned Shape: (2521, 4)

Task Distribution:
task_type
classification    2259
general            165
pitch_analysis      37
rewrite             30
critique            15
investor_eval       15
Name: count, dtype: int64

Average Input Length: 121.48
Average Output Length: 16.94


In [ ]:
import pandas as pd
import random

random.seed(42)

rows = []


weak_inputs = [
    "Need funding",
    "Large market opportunity",
    "We help everyone",
    "Great team",
    "We have users",
    "Good traction",
    "Strong growth",
    "Unique product",
    "We solve a problem",
    "Subscription model"
]

strong_outputs = [
    "Raising $500k to scale GTM, hire engineers, and expand pilots.",
    "$12B global market growing at 18% CAGR with underserved SME demand.",
    "Focused on Indian SMEs with 20–200 employees in retail and logistics.",
    "Founding team from Google, IIT, and McKinsey with domain expertise.",
    "12,000 monthly active users with 81% retention.",
    "$40k MRR growing 11% month-over-month.",
    "Revenue increased 4.2x in the last 12 months.",
    "Patent-pending AI engine reducing manual work by 60%.",
    "Cuts reconciliation time from 8 hours to 20 minutes weekly.",
    "$29/month SaaS pricing with enterprise annual contracts."
]

for _ in range(1200):
    i = random.randint(0, len(weak_inputs)-1)

    rows.append({
        "instruction":"Rewrite weak startup statement",
        "input": weak_inputs[i],
        "output": strong_outputs[i]
    })



biz_inputs = [
    "Free app with no monetization",
    "One-time payment only",
    "Subscription SaaS for SMEs",
    "Marketplace takes 10% commission",
    "Depends on ads only",
    "Enterprise annual contracts",
    "Freemium with paid upgrades",
    "High CAC low retention",
    "Single customer gives 90% revenue",
    "No pricing strategy"
]

biz_outputs = [
    "Weak model unless monetization roadmap is defined.",
    "Revenue may be limited without upsell or recurring streams.",
    "Strong recurring revenue model with scalable margins.",
    "Healthy if supply-demand liquidity is achieved.",
    "Risky unless user scale is massive.",
    "Strong B2B model with predictable recurring revenue.",
    "Good acquisition strategy if free-to-paid conversion is healthy.",
    "Unsustainable unit economics without retention improvement.",
    "High concentration risk. Diversify customer base.",
    "Pricing ambiguity reduces investor confidence."
]

for _ in range(1200):
    i = random.randint(0, len(biz_inputs)-1)

    rows.append({
        "instruction":"Evaluate business model",
        "input": biz_inputs[i],
        "output": biz_outputs[i]
    })


df = pd.DataFrame(rows)

df.to_csv("extra_startup_tasks.csv", index=False)

print(df.shape)
print(df["instruction"].value_counts())
print(df.head())

(2400, 3)
instruction
Rewrite weak startup statement    1200
Evaluate business model           1200
Name: count, dtype: int64
                      instruction                     input  \
0  Rewrite weak startup statement  Large market opportunity   
1  Rewrite weak startup statement              Need funding   
2  Rewrite weak startup statement             We have users   
3  Rewrite weak startup statement                Great team   
4  Rewrite weak startup statement                Great team   

                                              output  
0  $12B global market growing at 18% CAGR with un...  
1  Raising $500k to scale GTM, hire engineers, an...  
2    12,000 monthly active users with 81% retention.  
3  Founding team from Google, IIT, and McKinsey w...  
4  Founding team from Google, IIT, and McKinsey w...  


In [ ]:
main = pd.read_csv("final_training_dataset.csv")
extra = pd.read_csv("extra_startup_tasks.csv")

merged = pd.concat([main, extra], ignore_index=True)

merged = merged.sample(frac=1, random_state=42)

merged.to_csv("balanced_training_dataset.csv", index=False)

print(merged["instruction"].value_counts())

instruction
Classify sentiment of this financial statement                         2259
Evaluate business model                                                1215
Rewrite weak startup statement                                         1215
Analyze this startup pitch deck slide and provide investor feedback      22
Critique startup fundraising plan                                        15
Evaluate this Shark Tank startup pitch                                   15
Strengthen startup pitch slide                                           15
Provide VC feedback                                                      15
Review startup pitch                                                     15
Assess startup moat                                                      15
Analyze fundraising strategy                                             15
Assess startup valuation                                                 15
Check market attractiveness                                              15


In [ ]:
import pandas as pd



df = pd.read_csv("balanced_training_dataset.csv")

print("Before Balance:")
print(df["instruction"].value_counts())



target = {
    "Classify sentiment of this financial statement": 700,

    "Rewrite weak startup statement": 1500,
    "Evaluate business model": 1500,

    "Analyze this startup pitch deck slide and provide investor feedback": 1200,

    "Evaluate this Shark Tank startup pitch": 500
}

default_other = 500



parts = []

for instr in df["instruction"].unique():

    subset = df[df["instruction"] == instr]

    desired = target.get(instr, default_other)

    # oversample if needed
    if len(subset) < desired:
        subset = subset.sample(
            desired,
            replace=True,
            random_state=42
        )

    # downsample if too many
    else:
        subset = subset.sample(
            desired,
            random_state=42
        )

    parts.append(subset)


final_df = pd.concat(parts, ignore_index=True)


final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)


final_df.to_csv("final_balanced_dataset.csv", index=False)

print("\nAfter Balance:")
print(final_df["instruction"].value_counts())

print("\nFinal Shape:", final_df.shape)

Before Balance:
instruction
Classify sentiment of this financial statement                         2259
Evaluate business model                                                1215
Rewrite weak startup statement                                         1215
Analyze this startup pitch deck slide and provide investor feedback      22
Critique startup fundraising plan                                        15
Evaluate this Shark Tank startup pitch                                   15
Strengthen startup pitch slide                                           15
Provide VC feedback                                                      15
Review startup pitch                                                     15
Assess startup moat                                                      15
Analyze fundraising strategy                                             15
Assess startup valuation                                                 15
Check market attractiveness                                 

####DATASET SPLIT + BASELINE MODEL TRAINING  

In [ ]:


import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("final_balanced_dataset.csv")

print("Full Dataset Shape:", df.shape)



train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["instruction"]
)



val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["instruction"]
)

train_df.to_csv("train_balanced.csv", index=False)
val_df.to_csv("val_balanced.csv", index=False)
test_df.to_csv("test_balanced.csv", index=False)

print("Saved Successfully!")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Full Dataset Shape: (11900, 3)
Saved Successfully!
Train: (9520, 3)
Val: (1190, 3)
Test: (1190, 3)


In [ ]:
# BASELINE QWEN MODEL ON TEST DATA

!pip install -q transformers accelerate pandas torch

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


df = pd.read_csv("test_balanced.csv")

print("Test Shape:", df.shape)

df = df.sample(50, random_state=42).reset_index(drop=True)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)



def build_prompt(row):
    return f"""
You are an expert startup investor, startup mentor, and pitch deck analyst.

Instruction:
{row['instruction']}

Input:
{row['input']}

Respond clearly and professionally.
"""

results = []

for i, row in df.iterrows():

    prompt = build_prompt(row)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

    results.append({
        "instruction": row["instruction"],
        "input": row["input"],
        "expected_output": row["output"],
        "baseline_prediction": pred
    })

    print(f"Done {i+1}/{len(df)}")


out_df = pd.DataFrame(results)
out_df.to_csv("baseline_balanced_outputs.csv", index=False)

print("Saved Successfully!")

Test Shape: (1190, 3)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Done 1/50
Done 2/50
Done 3/50
Done 4/50
Done 5/50
Done 6/50
Done 7/50
Done 8/50
Done 9/50
Done 10/50
Done 11/50
Done 12/50
Done 13/50
Done 14/50
Done 15/50
Done 16/50
Done 17/50
Done 18/50
Done 19/50
Done 20/50
Done 21/50
Done 22/50
Done 23/50
Done 24/50
Done 25/50
Done 26/50
Done 27/50
Done 28/50
Done 29/50
Done 30/50
Done 31/50
Done 32/50
Done 33/50
Done 34/50
Done 35/50
Done 36/50
Done 37/50
Done 38/50
Done 39/50
Done 40/50
Done 41/50
Done 42/50
Done 43/50
Done 44/50
Done 45/50
Done 46/50
Done 47/50
Done 48/50
Done 49/50
Done 50/50
Saved Successfully!


In [ ]:

import zipfile
import os

zip_path = "fine_tuned_model.zip"
extract_path = "./fine_tuned_model"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully!")
print("Files extracted to:", extract_path)

Unzipped successfully!
Files extracted to: ./fine_tuned_model


In [2]:
import pandas as pd
out_df = pd.read_csv("baseline_balanced_outputs.csv")
print(out_df.head(10))

                                         instruction  \
0                            Evaluate business model   
1                            Evaluate business model   
2  Analyze this startup pitch deck slide and prov...   
3                     Rewrite weak startup statement   
4                        Check market attractiveness   
5                                Provide VC feedback   
6                        Evaluate investor readiness   
7                           Assess startup valuation   
8                  Critique startup fundraising plan   
9     Classify sentiment of this financial statement   

                                               input  \
0                   Marketplace takes 10% commission   
1                   Marketplace takes 10% commission   
2  The Team Bob Carbone – CEO, Founder – Experien...   
3                                 Subscription model   
4             Our founders have no domain experience   
5       We are raising at $20M valuation pre-re

##finetune model  

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu121
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes evaluate rouge_score

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/nightly/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.9/767.9 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 77.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 31.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 106.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes evaluate rouge_score
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from trl import SFTTrainer



MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"



In [ ]:


train_df = pd.read_csv("train_balanced.csv")
val_df = pd.read_csv("val_balanced.csv")

def format_row(row):
    return f"""User:
Instruction: {row['instruction']}
Input: {row['input']}

Assistant:
{row['output']}"""

train_df["text"] = train_df.apply(format_row, axis=1)
val_df["text"] = val_df.apply(format_row, axis=1)

train_ds = Dataset.from_pandas(train_df[["text"]])
val_ds = Dataset.from_pandas(val_df[["text"]])


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)



model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)


In [ ]:


args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,

    learning_rate=2e-4,
    num_train_epochs=2,
    max_steps=300,


    logging_steps=20,


    eval_steps=200,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=args
)

trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/9520 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9520 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1190 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1190 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,3.865752
40,2.572791
60,2.025102
80,1.876324
100,1.417479
120,1.491013
140,1.238874
160,1.367499
180,0.984462
200,0.961411


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=300, training_loss=1.529993673960368, metrics={'train_runtime': 237.2743, 'train_samples_per_second': 5.057, 'train_steps_per_second': 1.264, 'total_flos': 1425333281378304.0, 'train_loss': 1.529993673960368})

In [ ]:


trainer.model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")

('fine_tuned_model/tokenizer_config.json',
 'fine_tuned_model/chat_template.jinja',
 'fine_tuned_model/tokenizer.json')

In [ ]:
!zip -r fine_tuned_model.zip fine_tuned_model

  adding: fine_tuned_model/ (stored 0%)
  adding: fine_tuned_model/README.md (deflated 65%)
  adding: fine_tuned_model/tokenizer.json (deflated 81%)
  adding: fine_tuned_model/chat_template.jinja (deflated 71%)
  adding: fine_tuned_model/tokenizer_config.json (deflated 60%)
  adding: fine_tuned_model/adapter_model.safetensors (deflated 22%)
  adding: fine_tuned_model/adapter_config.json (deflated 58%)


In [ ]:


from google.colab import files

files.download("fine_tuned_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### TEST FINE-TUNED MODEL + COMPARE WITH BASELINE
### + QUANTITATIVE METRICS


In [ ]:
pip install -U peft transformers accelerate

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")  

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:


import zipfile
import os

zip_path = "fine_tuned_model.zip"
extract_path = "./fine_tuned_model"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully!")
print("Files extracted to:", extract_path)

In [ ]:
!pip uninstall -y torchao
!pip install torchao==0.16.0

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.6 MB/s eta 0:00:00


In [ ]:
from peft import PeftModel
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_PATH)

In [ ]:

# TEST FINE-TUNED MODEL + COMPARE WITH BASELINE



import pandas as pd
import torch
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32



df = pd.read_csv("test_balanced.csv")
print("Test Shape:", df.shape)

df = df.sample(100, random_state=42).reset_index(drop=True)


BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/fine_tuned_model/fine_tuned_model"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto"
)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto"
)

ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_PATH)

Test Shape: (1190, 3)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:




def generate_response(model, instruction, inp):

    prompt = f"""
User:
Instruction: {instruction}
Input: {inp}

Assistant:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return text.split("Assistant:")[-1].strip()



rows = []

for i, row in df.iterrows():

    instr = row["instruction"]
    inp   = row["input"]
    ref   = row["output"]

    pred_base = generate_response(base_model, instr, inp)
    pred_ft   = generate_response(ft_model, instr, inp)

    rows.append({
        "instruction": instr,
        "input": inp,
        "expected_output": ref,
        "baseline_prediction": pred_base,
        "finetuned_prediction": pred_ft
    })

    print(f"Done {i+1}/{len(df)}")

res_df = pd.DataFrame(rows)
res_df.to_csv("comparison_outputs.csv", index=False)

print("Saved comparison_outputs.csv")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Done 1/100
Done 2/100
Done 3/100
Done 4/100
Done 5/100
Done 6/100
Done 7/100
Done 8/100
Done 9/100
Done 10/100
Done 11/100
Done 12/100
Done 13/100
Done 14/100
Done 15/100
Done 16/100
Done 17/100
Done 18/100
Done 19/100
Done 20/100
Done 21/100
Done 22/100
Done 23/100
Done 24/100
Done 25/100
Done 26/100
Done 27/100
Done 28/100
Done 29/100
Done 30/100
Done 31/100
Done 32/100
Done 33/100
Done 34/100
Done 35/100
Done 36/100
Done 37/100
Done 38/100
Done 39/100
Done 40/100
Done 41/100
Done 42/100
Done 43/100
Done 44/100
Done 45/100
Done 46/100
Done 47/100
Done 48/100
Done 49/100
Done 50/100
Done 51/100
Done 52/100
Done 53/100
Done 54/100
Done 55/100
Done 56/100
Done 57/100
Done 58/100
Done 59/100
Done 60/100
Done 61/100
Done 62/100
Done 63/100
Done 64/100
Done 65/100
Done 66/100
Done 67/100
Done 68/100
Done 69/100
Done 70/100
Done 71/100
Done 72/100
Done 73/100
Done 74/100
Done 75/100
Done 76/100
Done 77/100
Done 78/100
Done 79/100
Done 80/100
Done 81/100
Done 82/100
Done 83/100
Done 84/100
D

In [ ]:


# METRICS

refs = res_df["expected_output"].tolist()
pred_base = res_df["baseline_prediction"].tolist()
pred_ft   = res_df["finetuned_prediction"].tolist()



def simple_tokenize(text):
    return text.lower().split()




def bleu_score(preds, refs):

    scores = []

    for p, r in zip(preds, refs):
        p_tokens = simple_tokenize(p)
        r_tokens = simple_tokenize(r)

        overlap = len(set(p_tokens) & set(r_tokens))
        score = overlap / max(len(p_tokens), 1)

        scores.append(score)

    return sum(scores) / len(scores)

bleu_base = bleu_score(pred_base, refs)
bleu_ft   = bleu_score(pred_ft, refs)



def compute_rouge(preds, refs):

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        scores["rouge1"].append(s["rouge1"].fmeasure)
        scores["rouge2"].append(s["rouge2"].fmeasure)
        scores["rougeL"].append(s["rougeL"].fmeasure)

    return {k: sum(v)/len(v) for k,v in scores.items()}

rouge_base = compute_rouge(pred_base, refs)
rouge_ft   = compute_rouge(pred_ft, refs)


In [ ]:

# FINANCE ACCURACY


def extract_label(text):

    t = text.lower()

    if "positive" in t:
        return "positive"
    elif "negative" in t:
        return "negative"
    else:
        return "neutral"

finance_df = res_df[
    res_df["instruction"] ==
    "Classify sentiment of this financial statement"
]

acc_base, acc_ft = 0, 0

if len(finance_df) > 0:

    base_labels = finance_df["baseline_prediction"].apply(extract_label)
    ft_labels   = finance_df["finetuned_prediction"].apply(extract_label)
    true_labels = finance_df["expected_output"]

    acc_base = (base_labels == true_labels).mean()
    acc_ft   = (ft_labels == true_labels).mean()

In [ ]:




with open("metrics_report.txt", "w") as f:

    f.write("===== MODEL COMPARISON REPORT =====\n\n")

    f.write("BLEU SCORE\n")
    f.write(f"Baseline   : {bleu_base:.4f}\n")
    f.write(f"Fine-Tuned : {bleu_ft:.4f}\n\n")

    f.write("ROUGE SCORE\n")
    f.write(f"Baseline   : {rouge_base}\n")
    f.write(f"Fine-Tuned : {rouge_ft}\n\n")

    f.write("FINANCE SENTIMENT ACCURACY\n")
    f.write(f"Baseline   : {acc_base:.4f}\n")
    f.write(f"Fine-Tuned : {acc_ft:.4f}\n\n")

print("Saved metrics_report.txt")

Saved metrics_report.txt


In [ ]:


print("\n===== FINAL RESULTS =====")
print("BLEU Baseline   :", round(bleu_base,4))
print("BLEU Finetuned  :", round(bleu_ft,4))

print("Finance Acc Baseline :", round(acc_base,4))
print("Finance Acc Finetune :", round(acc_ft,4))

print("\nROUGE Baseline :", rouge_base)
print("ROUGE Finetune:", rouge_ft)


===== FINAL RESULTS =====
BLEU Baseline   : 0.0225
BLEU Finetuned  : 0.2099
Finance Acc Baseline : 0.4
Finance Acc Finetune : 0.6

ROUGE Baseline : {'rouge1': 0.0583985804343326, 'rouge2': 0.0026429277074192094, 'rougeL': 0.04132335736319041}
ROUGE Finetune: {'rouge1': 0.25837680636280325, 'rouge2': 0.11515198903641206, 'rougeL': 0.23473021131247016}


In [ ]:
import csv

res_df["baseline_prediction"] = res_df["baseline_prediction"].str.replace("\n", " ", regex=False)
res_df["baseline_prediction"] = res_df["baseline_prediction"].str.replace("\r", " ", regex=False)

res_df.to_csv(
    "baseline_balanced_outputs_clean.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

In [ ]:
# PROMPT ENGINEERING BASELINE COMPARISON


import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel



BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_PATH = "/content/fine_tuned_model/fine_tuned_model"
TEST_FILE = "/content/test_balanced.csv"
OUTPUT_FILE = "prompt_engineering_comparison.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32
).to(DEVICE)


ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32
).to(DEVICE)

fine_tuned_model = PeftModel.from_pretrained(ft_base, LORA_PATH)


def generate(model, prompt):

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return text[len(prompt):].strip()



df = pd.read_csv(TEST_FILE)

df = df.head(20)

results = []


for i, row in df.iterrows():

    instruction = str(row["instruction"])
    inp = str(row["input"])

    

    raw_prompt = f"""
Instruction: {instruction}

Input: {inp}

Response:
"""

    raw_output = generate(base_model, raw_prompt)

    

    engineered_prompt = f"""
You are an expert startup investor, pitch deck analyst, and venture capitalist.

Analyze the startup content professionally.

Return:
1. Strengths
2. Weaknesses
3. Suggestions

Use concise bullet points.

Task: {instruction}

Startup Content:
{inp}

Answer:
"""

    engineered_output = generate(base_model, engineered_prompt)



    ft_prompt = f"""
Instruction: {instruction}

Input: {inp}

Response:
"""

    ft_output = generate(fine_tuned_model, ft_prompt)

   
    results.append({
        "instruction": instruction,
        "input": inp,
        "raw_baseline_output": raw_output,
        "prompt_engineered_output": engineered_output,
        "fine_tuned_output": ft_output
    })

    print(f"Done {i+1}/20")



pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)
print("Completed Prompt Engineering Comparison.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Done 1/20
Done 2/20
Done 3/20
Done 4/20
Done 5/20
Done 6/20
Done 7/20
Done 8/20
Done 9/20
Done 10/20
Done 11/20
Done 12/20
Done 13/20
Done 14/20
Done 15/20
Done 16/20
Done 17/20
Done 18/20
Done 19/20
Done 20/20

Saved: prompt_engineering_comparison.csv
Completed Prompt Engineering Comparison.


In [9]:
pip install rouge_score

In [ ]:

import pandas as pd
from rouge_score import rouge_scorer


df = pd.read_csv("/content/prompt_engineering_comparison.csv")



scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

r1 = []
r2 = []
rL = []
lengths = []



for _, row in df.iterrows():

    reference = str(row["fine_tuned_output"])
    pred = str(row["prompt_engineered_output"])

    score = scorer.score(reference, pred)

    r1.append(score["rouge1"].fmeasure)
    r2.append(score["rouge2"].fmeasure)
    rL.append(score["rougeL"].fmeasure)

    lengths.append(len(pred.split()))



print("Prompt Engineered Column Values")
print("-------------------------------")
print("avg_rouge1 :", round(sum(r1)/len(r1), 4))
print("avg_rouge2 :", round(sum(r2)/len(r2), 4))
print("avg_rougeL :", round(sum(rL)/len(rL), 4))
print("avg_response_length :", round(sum(lengths)/len(lengths), 1))

Prompt Engineered Column Values
-------------------------------
avg_rouge1 : 0.1934
avg_rouge2 : 0.0285
avg_rougeL : 0.0978
avg_response_length : 150.4
